# 🌿 HUẤN LUYỆN & ĐÁNH GIÁ MÔ HÌNH NHẬN DIỆN 90 LOÀI CÂY KIỂNG LÁ (TỶ LỆ 80% - 10% - 10%)
**Tự động lấy bộ dữ liệu gốc `merged_90_split.zip` (23.850 ảnh - 90 loài) trên Google Drive, chia lại đúng tỷ lệ `80% Train - 10% Val - 10% Test`, huấn luyện lại từ đầu (2 giai đoạn) và xuất đầy đủ Ma trận nhầm lẫn + Precision, Recall, F1-Score, Accuracy.**

### Hướng dẫn chạy:
1. Vào menu **Runtime (Thời gian chạy)** -> **Change runtime type (Thay đổi loại thời gian chạy)** -> Chọn **T4 GPU**.
2. Bấm **Runtime (Thời gian chạy)** -> **Run all (Chạy tất cả)** (hoặc bấm nút Play ▶ ở từng ô từ trên xuống dưới).
3. Toàn bộ tiến độ sau mỗi Epoch và báo cáo kết quả sẽ tự động lưu vào Google Drive tại thư mục: `MyDrive/KiengLa_AI/train_90_lop_80_10_10/`.

In [ ]:
# === BƯỚC 1: KẾT NỐI GOOGLE DRIVE & GIẢI NÉN BỘ DỮ LIỆU GỐC (23.850 ẢNH) ===
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile
import tensorflow as tf

PROJECT = Path('/content/drive/MyDrive/KiengLa_AI')
ZIP_PATH = PROJECT / 'merged_v1/merged_90_split.zip'
RAW_EXTRACT_DIR = Path('/content/merged_90_raw')

assert ZIP_PATH.is_file(), f'Không tìm thấy file dữ liệu gốc: {ZIP_PATH}'
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), '⚠️ Hãy bật GPU trong Runtime -> Change runtime type -> T4 GPU!'

if not (RAW_EXTRACT_DIR / '_extracted_ok').exists():
    RAW_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print('⏳ Đang giải nén merged_90_split.zip (6.2 GB)... Vui lòng đợi khoảng 30-45 giây...')
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(RAW_EXTRACT_DIR)
    (RAW_EXTRACT_DIR / '_extracted_ok').touch()

print('✅ Đã giải nén xong dữ liệu gốc tại:', RAW_EXTRACT_DIR)


In [ ]:
# === BƯỚC 2: GOM TOÀN BỘ 23.850 ẢNH VÀ CHIA LẠI ĐÚNG TỶ LỆ 80% TRAIN - 10% VAL - 10% TEST ===
import os
import shutil
import random
import pandas as pd
from pathlib import Path

DATA_DIR = Path('/content/merged_90_split_80_10_10')
SAVE_ROOT = PROJECT / 'train_90_lop_80_10_10'
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

if not (DATA_DIR / '_split_80_10_10_ok').exists():
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)

    # Lấy danh sách 90 lớp từ thư mục gốc
    class_folders = sorted([d.name for d in (RAW_EXTRACT_DIR / 'train').iterdir() if d.is_dir()])
    assert len(class_folders) == 90, f'Số lớp không đúng 90: {len(class_folders)}'

    rng = random.Random(42)
    split_rows = []

    for cls_name in class_folders:
        # Gom tất cả ảnh của loài này từ cả 3 tập cũ (train, val, test)
        all_imgs = []
        for old_split in ['train', 'val', 'test']:
            cls_dir = RAW_EXTRACT_DIR / old_split / cls_name
            if cls_dir.is_dir():
                all_imgs.extend([
                    p for p in cls_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in EXTENSIONS
                ])

        # Sắp xếp theo tên file (hash) rồi xáo trộn cố định với seed=42
        all_imgs = sorted(all_imgs, key=lambda p: p.name)
        rng.shuffle(all_imgs)

        n = len(all_imgs)
        n_train = int(n * 0.80)
        n_val = int(n * 0.90) - n_train
        # Đảm bảo ít nhất 1 ảnh val và 1 ảnh test
        if n_val == 0 and n >= 3:
            n_val = 1

        splits = {
            'train': all_imgs[:n_train],
            'val': all_imgs[n_train:n_train + n_val],
            'test': all_imgs[n_train + n_val:]
        }

        for split_name, paths in splits.items():
            dest_dir = DATA_DIR / split_name / cls_name
            dest_dir.mkdir(parents=True, exist_ok=True)
            for src_path in paths:
                dst_path = dest_dir / src_path.name
                try:
                    os.link(src_path, dst_path)
                except OSError:
                    shutil.copy2(src_path, dst_path)

        split_rows.append({
            'label': cls_name,
            'train': len(splits['train']),
            'val': len(splits['val']),
            'test': len(splits['test']),
            'total': n
        })

    split_df = pd.DataFrame(split_rows)
    split_df.to_csv(SAVE_ROOT / 'split_counts_80_10_10.csv', index=False, encoding='utf-8-sig')
    (DATA_DIR / '_split_80_10_10_ok').touch()
else:
    split_df = pd.read_csv(SAVE_ROOT / 'split_counts_80_10_10.csv')

tot_train = int(split_df['train'].sum())
tot_val = int(split_df['val'].sum())
tot_test = int(split_df['test'].sum())
tot_all = int(split_df['total'].sum())

print('=' * 68)
print('✅ KẾT QUẢ CHIA LẠI TẬP DỮ LIỆU THEO TỶ LỆ 80% - 10% - 10%')
print('=' * 68)
print(f'Tổng số loài cây : {len(split_df)} loài')
print(f'Tổng số ảnh gốc  : {tot_all:,} ảnh (100%)')
print(f'  + Tập TRAIN    : {tot_train:,} ảnh ({tot_train / tot_all:.2%})')
print(f'  + Tập VAL      : {tot_val:,} ảnh ({tot_val / tot_all:.2%})')
print(f'  + Tập TEST     : {tot_test:,} ảnh ({tot_test / tot_all:.2%}) -> Độc lập hoàn toàn')
print('=' * 68)
print('Đã lưu bảng thống kê tại:', SAVE_ROOT / 'split_counts_80_10_10.csv')
display(split_df.head(10))


In [ ]:
# === BƯỚC 3: NẠP DỮ LIỆU TRAIN / VAL / TEST VÀ TÍNH TRỌNG SỐ CÂN BẰNG LỚP ===
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight

keras.utils.set_random_seed(42)

def load_dataset(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        str(DATA_DIR / split),
        image_size=(224, 224),
        interpolation='nearest',
        batch_size=32,
        label_mode='int',
        shuffle=shuffle,
        seed=42
    )

train_ds_raw = load_dataset('train', True)
val_ds_raw = load_dataset('val', False)
test_ds_raw = load_dataset('test', False)

class_names = train_ds_raw.class_names
assert class_names == val_ds_raw.class_names == test_ds_raw.class_names
assert len(class_names) == 90, f'Đang có {len(class_names)} lớp!'

# Tính Balanced Class Weights để các loài ít ảnh không bị lép vế
y_train_all = np.concatenate([y.numpy() for _, y in train_ds_raw], axis=0)
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(class_names)),
    y=y_train_all
)
class_weight_dict = {i: float(w) for i, w in enumerate(weights)}

train_ds = train_ds_raw.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds_raw.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds_raw.prefetch(tf.data.AUTOTUNE)

SAVE_DIR = SAVE_ROOT / 'head'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CLASS_PATH = SAVE_ROOT / 'class_names.json'
CLASS_PATH.write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
(SAVE_DIR / 'class_names.json').write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')

print('✅ Đã chuẩn bị xong Data Pipeline cho 90 lớp.')
print('Số mẫu huấn luyện (Train):', len(y_train_all))


In [ ]:
# === BƯỚC 4: HUẤN LUYỆN GIAI ĐOẠN 1 (TRANSFER LEARNING - ĐÓNG BĂNG BACKBONE MOBILENETV2) ===
import math
import os

STATE_PATH = SAVE_DIR / 'progress.json'
BEST_PATH = SAVE_DIR / 'best_model.keras'

class Save90(keras.callbacks.Callback):
    def __init__(self, save_dir, state_path, best_path, patience=6, min_delta=0.001):
        super().__init__()
        self.save_dir = save_dir
        self.state_path = state_path
        self.best_path = best_path
        self.patience = patience
        self.min_delta = min_delta

        if self.state_path.exists():
            self.state = json.loads(self.state_path.read_text(encoding='utf-8'))
        else:
            self.state = {
                'next_epoch': 0,
                'best_loss': None,
                'reference_loss': None,
                'wait': 0,
                'stopped': False,
                'checkpoint': None
            }

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = float(logs['val_loss'])

        if not math.isfinite(loss):
            print('Sai số không hợp lệ; dừng.')
            self.model.stop_training = True
            return

        if self.state['best_loss'] is None or loss < self.state['best_loss']:
            temp = self.save_dir / 'best_pending.keras'
            self.model.save(str(temp))
            os.replace(temp, self.best_path)
            self.state['best_loss'] = loss

        reference = self.state['reference_loss']
        if reference is None or loss < reference - self.min_delta:
            self.state['reference_loss'] = loss
            self.state['wait'] = 0
        else:
            self.state['wait'] += 1

        checkpoint = self.save_dir / f'epoch_{epoch + 1:03d}.keras'
        self.model.save(str(checkpoint))

        self.state.update({
            'next_epoch': epoch + 1,
            'checkpoint': checkpoint.name,
            'stopped': self.state['wait'] >= self.patience,
            'last_metrics': {key: float(value) for key, value in logs.items()}
        })

        temp = self.save_dir / 'progress_pending.json'
        temp.write_text(json.dumps(self.state, indent=2), encoding='utf-8')
        os.replace(temp, self.state_path)

        for old in sorted(self.save_dir.glob('epoch_*.keras'))[:-2]:
            old.unlink()

        print(f"\nĐã lưu epoch {epoch + 1} vào Drive. Không cải thiện: {self.state['wait']}/{self.patience}")

        if self.state['stopped']:
            print('Giai đoạn này đã bão hòa validation loss.')
            self.model.stop_training = True

if STATE_PATH.exists():
    state = json.loads(STATE_PATH.read_text(encoding='utf-8'))
    model = keras.models.load_model(str(SAVE_DIR / state['checkpoint']))
    initial_epoch = state['next_epoch']
    print('Khôi phục Giai đoạn 1 từ epoch:', initial_epoch)
else:
    base_model = keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = layers.RandomFlip('horizontal')(inputs)
    x = layers.RandomRotation(0.08)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(class_names), activation='softmax', name='classifier_90')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy')
        ]
    )
    initial_epoch = 0
    print('Khởi tạo mới model MobileNetV2 (90 lớp) cho tỷ lệ 80-10-10.')

save_cb_head = Save90(SAVE_DIR, STATE_PATH, BEST_PATH, patience=6, min_delta=0.001)
if save_cb_head.state['stopped']:
    print('✅ Giai đoạn 1 (Head) đã hoàn tất trước đó.')
else:
    history_head = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=initial_epoch,
        epochs=initial_epoch + 40,
        class_weight=class_weight_dict,
        callbacks=[
            keras.callbacks.TerminateOnNaN(),
            save_cb_head,
            keras.callbacks.CSVLogger(str(SAVE_DIR / 'training_log.csv'), append=True)
        ]
    )


In [ ]:
# === BƯỚC 5: HUẤN LUYỆN GIAI ĐOẠN 2 (FINE-TUNING 30 LỚP CUỐI MOBILENETV2 VỚI LR = 1e-5) ===
HEAD_DIR = SAVE_ROOT / 'head'
FINE_DIR = SAVE_ROOT / 'finetune'
FINE_DIR.mkdir(parents=True, exist_ok=True)

FINE_STATE_PATH = FINE_DIR / 'progress.json'
FINE_BEST_PATH = FINE_DIR / 'best_model.keras'
(FINE_DIR / 'class_names.json').write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8'
)

if FINE_STATE_PATH.exists():
    state_fine = json.loads(FINE_STATE_PATH.read_text(encoding='utf-8'))
    model = keras.models.load_model(str(FINE_DIR / state_fine['checkpoint']))
    initial_epoch = state_fine['next_epoch']
    print('Khôi phục Fine-tune từ epoch:', initial_epoch)
else:
    model = keras.models.load_model(str(HEAD_DIR / 'best_model.keras'))
    baseline = model.evaluate(val_ds, return_dict=True, verbose=0)
    baseline_loss = float(baseline['loss'])
    print('Validation trước khi Fine-tune:', baseline)

    base_model = next(
        layer for layer in model.layers
        if isinstance(layer, keras.Model) and 'mobilenet' in layer.name.lower()
    )
    base_model.trainable = True

    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        layer.trainable = not isinstance(layer, layers.BatchNormalization)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy')
        ]
    )

    model.save(str(FINE_BEST_PATH))
    checkpoint = FINE_DIR / 'epoch_000.keras'
    model.save(str(checkpoint))

    state_fine = {
        'next_epoch': 0,
        'best_loss': baseline_loss,
        'reference_loss': baseline_loss,
        'wait': 0,
        'stopped': False,
        'checkpoint': checkpoint.name
    }
    FINE_STATE_PATH.write_text(json.dumps(state_fine, indent=2), encoding='utf-8')
    initial_epoch = 0

save_cb_fine = Save90(FINE_DIR, FINE_STATE_PATH, FINE_BEST_PATH, patience=6, min_delta=0.001)
if save_cb_fine.state['stopped']:
    print('✅ Giai đoạn 2 (Fine-tune) đã hoàn tất trước đó.')
else:
    history_fine = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=initial_epoch,
        epochs=initial_epoch + 50,
        class_weight=class_weight_dict,
        callbacks=[
            keras.callbacks.TerminateOnNaN(),
            save_cb_fine,
            keras.callbacks.CSVLogger(str(FINE_DIR / 'training_log.csv'), append=True)
        ]
    )


In [ ]:
# === BƯỚC 6: ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST ĐỘC LẬP (10% ~ 2.400 ẢNH) & XUẤT CONFUSION MATRIX ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

OUT_DIR = FINE_DIR / 'evaluation_report_80_10_10'
OUT_DIR.mkdir(parents=True, exist_ok=True)

best_model = keras.models.load_model(str(FINE_BEST_PATH), compile=False)

true_batches, score_batches = [], []
for images, labels in test_ds:
    scores = best_model(images, training=False).numpy()
    true_batches.append(labels.numpy())
    score_batches.append(scores)

y_true = np.concatenate(true_batches)
scores_all = np.concatenate(score_batches)
y_pred = scores_all.argmax(axis=1)
class_ids = np.arange(len(class_names))

# Đọc tên tiếng Việt từ Drive
vi_map = {}
labels_path = PROJECT / 'labels/labels_vi.json'
if labels_path.exists():
    with open(labels_path, encoding='utf-8') as f:
        for row in json.load(f):
            vi_map[row['original_label']] = row['vietnamese_name']

display_names = [f"{i:02d} - {vi_map.get(name, name)}" for i, name in enumerate(class_names)]

total = int(len(y_true))
correct = int(np.sum(y_true == y_pred))
incorrect = total - correct

top1_acc = float(accuracy_score(y_true, y_pred))
top3_acc = float(np.mean(np.any(np.argsort(scores_all, axis=1)[:, -3:] == y_true[:, None], axis=1)))
macro_prec = float(precision_score(y_true, y_pred, labels=class_ids, average='macro', zero_division=0))
macro_rec = float(recall_score(y_true, y_pred, labels=class_ids, average='macro', zero_division=0))
macro_f1 = float(f1_score(y_true, y_pred, labels=class_ids, average='macro', zero_division=0))
weighted_prec = float(precision_score(y_true, y_pred, labels=class_ids, average='weighted', zero_division=0))
weighted_rec = float(recall_score(y_true, y_pred, labels=class_ids, average='weighted', zero_division=0))
weighted_f1 = float(f1_score(y_true, y_pred, labels=class_ids, average='weighted', zero_division=0))

report_text = classification_report(
    y_true, y_pred,
    labels=class_ids,
    target_names=display_names,
    digits=4,
    zero_division=0
)

cm = confusion_matrix(y_true, y_pred, labels=class_ids)

summary = (
    '=' * 72 + '\n'
    'KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP (TỶ LỆ 80% - 10% - 10%)\n'
    + '=' * 72 + '\n'
    f'Tổng số ảnh Test (10%) : {total}\n'
    f'Dự đoán ĐÚNG (TP)      : {correct}\n'
    f'Dự đoán SAI            : {incorrect}\n'
    f'TEST ACCURACY (Top-1)  : {top1_acc:.2%}\n'
    f'TOP-3 ACCURACY         : {top3_acc:.2%}\n'
    f'MACRO PRECISION        : {macro_prec:.2%} (Weighted: {weighted_prec:.2%})\n'
    f'MACRO RECALL           : {macro_rec:.2%} (Weighted: {weighted_rec:.2%})\n'
    f'MACRO F1-SCORE         : {macro_f1:.2%} (Weighted: {weighted_f1:.2%})\n'
    + '=' * 72 + '\n\n'
    'CLASSIFICATION REPORT (CHI TIẾT 90 LOÀI):\n'
    + report_text
)

print(summary)

# Lưu các file báo cáo
with open(OUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump({
        'split_ratio': '80-10-10',
        'test_images': total,
        'correct_predictions': correct,
        'incorrect_predictions': incorrect,
        'accuracy': top1_acc,
        'top3_accuracy': top3_acc,
        'macro_precision': macro_prec,
        'macro_recall': macro_rec,
        'macro_f1': macro_f1,
        'weighted_precision': weighted_prec,
        'weighted_recall': weighted_rec,
        'weighted_f1': weighted_f1
    }, f, ensure_ascii=False, indent=2)

report_dict = classification_report(
    y_true, y_pred, labels=class_ids, target_names=class_names, output_dict=True, zero_division=0
)
pd.DataFrame(report_dict).T.to_csv(OUT_DIR / 'classification_report.csv', encoding='utf-8-sig')

cm_df_en = pd.DataFrame(cm, index=class_names, columns=class_names)
cm_df_en.to_csv(OUT_DIR / 'confusion_matrix.csv', encoding='utf-8-sig')

cm_df_vi = pd.DataFrame(cm, index=display_names, columns=display_names)
cm_df_vi.index.name = 'Thực tế'
cm_df_vi.columns.name = 'Dự đoán'
cm_df_vi.to_csv(OUT_DIR / 'confusion_matrix_vi.csv', encoding='utf-8-sig')

with open(OUT_DIR / 'evaluation_summary_vi.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
    f.write('\n\nCONFUSION MATRIX — HÀNG: THỰC TẾ, CỘT: DỰ ĐOÁN\n')
    f.write(cm_df_vi.to_string())

# Vẽ Ma trận nhầm lẫn 90x90
fig, ax = plt.subplots(figsize=(22, 20))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_ids).plot(
    ax=ax, cmap='Blues', include_values=False, xticks_rotation=90, colorbar=True
)
ax.set_title('Ma trận nhầm lẫn — 90 lớp cây (Tỷ lệ 80% - 10% - 10%)')
ax.set_xlabel('Nhãn dự đoán (mã lớp)')
ax.set_ylabel('Nhãn thật (mã lớp)')
ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.savefig(OUT_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)

# Xuất danh sách các cặp loài bị nhầm nhiều nhất
errors = [
    {
        'Cây thật': display_names[i],
        'Bị đoán thành': display_names[j],
        'Số ảnh nhầm': int(cm[i, j]),
        'Tổng ảnh lớp': int(cm[i].sum()),
        'Tỷ lệ nhầm': float(cm[i, j] / cm[i].sum())
    }
    for i in class_ids for j in class_ids if i != j and cm[i, j] > 0
]
if errors:
    errors_df = pd.DataFrame(errors).sort_values('Số ảnh nhầm', ascending=False)
    errors_df.to_csv(OUT_DIR / 'confused_pairs_vi.csv', index=False, encoding='utf-8-sig')
    print('\n20 cặp loài bị nhầm nhiều nhất:')
    display(errors_df.head(20))

print('\n✅ Đã lưu toàn bộ báo cáo đánh giá vào Google Drive tại:', OUT_DIR)


In [ ]:
# === BƯỚC 7: ĐÓNG GÓI MODEL MỚI + TOÀN BỘ BÁO CÁO ĐỂ TẢI VỀ MÁY ===
from google.colab import files
import zipfile

zip_out = Path('/content/KiengLa_Model_Va_BaoCao_80_10_10.zip')
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(FINE_BEST_PATH, arcname='best_model.keras')
    z.write(FINE_DIR / 'class_names.json', arcname='class_names.json')
    if (PROJECT / 'labels/labels_vi.json').exists():
        z.write(PROJECT / 'labels/labels_vi.json', arcname='labels_vi.json')
    z.write(SAVE_ROOT / 'split_counts_80_10_10.csv', arcname='split_counts_80_10_10.csv')
    for report_file in OUT_DIR.iterdir():
        if report_file.is_file():
            z.write(report_file, arcname=f'evaluation_report_80_10_10/{report_file.name}')

# Lưu 1 bản zip dự phòng lên Google Drive
shutil.copy2(zip_out, SAVE_ROOT / zip_out.name)
print('✅ Đã lưu bản sao trên Drive:', SAVE_ROOT / zip_out.name)
files.download(str(zip_out))
